In [3]:
# ============================================================
# MDSM Dataset Preparation — LOCAL ONLY
#
# Reads ALL JSON files in MDSM-public/meta/
# For each JSON file:
#   → extracts records  → <name>.csv
#   → finds referenced images → <name>-image/ folder
#
# Output structure:
#   OUTPUT_BASE/
#   ├── washington-post.csv
#   ├── washington-post-image/
#   ├── nytimes.csv
#   ├── nytimes-image/
#   └── ...
#
# Requirements:
#   pip install pandas tqdm
# ============================================================

import os, json, glob, shutil
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONFIGURE THESE BEFORE RUNNING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Full path to your MDSM-public folder
# Windows  → r"C:\Users\YourName\Downloads\MDSM-public"
# Mac/Linux → "/home/yourname/datasets/MDSM-public"
DATASET_ROOT = r"C:\Users\User\Downloads\MDSM-public\MDSM-public"   # ← CHANGE THIS

# Where you want the output saved
# Windows  → r"C:\Users\YourName\Documents\FYP\mdsm"
# Mac/Linux → "/home/yourname/FYP/mdsm"
OUTPUT_BASE = r"C:\Users\User\Downloads\FYP\mdsm"       # ← CHANGE THIS

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

META_DIR  = os.path.join(DATASET_ROOT, "meta")
IMAGE_DIR = os.path.join(DATASET_ROOT, "image")

MIME_EXTS = {".jpg", ".jpeg", ".png", ".gif", ".webp", ".bmp", ".tiff"}

os.makedirs(OUTPUT_BASE, exist_ok=True)

# ══════════════════════════════════════════════════════════════
#  HELPERS
# ══════════════════════════════════════════════════════════════

def parse_json_to_df(json_path):
    """Load a JSON file and return a flat DataFrame + raw data.

    Actual MDSM record shape:
      {"id": "MDSM_00407141", "image": "image/MDSM_00407141.jpg",
       "text": "...", "fake_cls": "orig", "fake_image_box": [...]}
    """
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    if isinstance(data, list):
        items = data
    elif isinstance(data, dict):
        items = data.get("articles", data.get("data", [data]))
    else:
        return pd.DataFrame(), data

    records = []
    for item in items:
        if not isinstance(item, dict):
            continue
        records.append({
            "id":             item.get("id", ""),
            # Store just the filename, not the full relative path
            "image":          Path(item.get("image", "")).name,
            "text":           item.get("text", ""),
            "fake_cls":       item.get("fake_cls", ""),
            "fake_image_box": json.dumps(item.get("fake_image_box", [])),
        })
    return pd.DataFrame(records), data


def collect_image_refs(data):
    """Collect image filenames from every record's 'image' field.

    The field value is a path like "image/MDSM_00407141.jpg".
    We only need the filename part: "MDSM_00407141.jpg".
    """
    refs = set()
    items = data if isinstance(data, list) else data.get("articles", data.get("data", [data]))
    for item in items:
        if not isinstance(item, dict):
            continue
        val = item.get("image", "")
        if val:
            # Strip any leading path (e.g. "image/") — keep only the filename
            refs.add(Path(val).name)
    return refs


def find_image_on_disk(filename, image_dir):
    """Look up a filename directly in the flat image folder."""
    candidate = Path(image_dir) / filename
    if candidate.is_file():
        return candidate
    # Fallback: stem match in case extension differs
    stem = Path(filename).stem
    for ext in MIME_EXTS:
        c = Path(image_dir) / f"{stem}{ext}"
        if c.is_file():
            return c
    return None


# ══════════════════════════════════════════════════════════════
#  MAIN
# ══════════════════════════════════════════════════════════════

print("=" * 60)
print("  MDSM DATA PREP — LOCAL")
print("=" * 60)
print(f"  Source : {DATASET_ROOT}")
print(f"  Output : {OUTPUT_BASE}")

# ── Validate source paths ─────────────────────────────────────
for p, label in [(META_DIR, "meta folder"), (IMAGE_DIR, "image folder")]:
    if not os.path.exists(p):
        raise FileNotFoundError(
            f"❌  {label} not found:\n    {p}\n"
            "    Check DATASET_ROOT at the top of this script."
        )

# ── Discover all JSON files ───────────────────────────────────
json_files = sorted(glob.glob(os.path.join(META_DIR, "*.json")))
print(f"\n🔍 Found {len(json_files)} JSON file(s) in meta/:")
for jf in json_files:
    print(f"   • {os.path.basename(jf)}")

if not json_files:
    raise FileNotFoundError(f"No JSON files found in {META_DIR}.")

# ── Process each JSON file ────────────────────────────────────
grand_total = {"records": 0, "img_refs": 0, "matched": 0,
               "copied": 0, "skipped": 0, "errors": 0, "missing": 0}

for json_path in json_files:
    source_name   = Path(json_path).stem              # e.g. "washington-post"
    csv_name      = f"{source_name}.csv"
    img_folder    = f"{source_name}-image"

    out_csv       = os.path.join(OUTPUT_BASE, csv_name)
    out_img_dir   = os.path.join(OUTPUT_BASE, img_folder)
    os.makedirs(out_img_dir, exist_ok=True)

    print(f"\n{'─'*60}")
    print(f"  📰 {os.path.basename(json_path)}")
    print(f"{'─'*60}")

    # ── 1. Parse JSON → DataFrame ─────────────────────────────
    try:
        df, raw_data = parse_json_to_df(json_path)
    except Exception as e:
        print(f"  ⚠️  Failed to parse: {e} — skipping.")
        continue

    print(f"  Records parsed : {len(df):,}")
    grand_total["records"] += len(df)

    # ── 2. Save CSV ───────────────────────────────────────────
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    csv_mb = os.path.getsize(out_csv) / 1_048_576
    print(f"  💾 Saved → {csv_name}  ({csv_mb:.2f} MB)")

    # ── 3. Collect + resolve image references ─────────────────
    image_refs = collect_image_refs(raw_data)
    print(f"  Image refs     : {len(image_refs):,}")
    grand_total["img_refs"] += len(image_refs)

    resolved = {}
    missing  = []
    for ref in image_refs:
        path = find_image_on_disk(ref, IMAGE_DIR)
        if path:
            resolved[ref] = path
        else:
            missing.append(ref)

    print(f"  Matched on disk: {len(resolved):,}")
    if missing:
        print(f"  ⚠️  Not found  : {len(missing):,}  (e.g. {missing[:3]})")
    grand_total["matched"] += len(resolved)
    grand_total["missing"] += len(missing)

    # ── 4. Copy images to output folder ───────────────────────
    copied = skipped = errors = 0
    for ref, src in tqdm(resolved.items(), desc=f"  Copying {source_name}", unit="img"):
        dst = Path(out_img_dir) / src.name
        if dst.exists():
            skipped += 1
            continue
        try:
            shutil.copy2(src, dst)
            copied += 1
        except Exception as e:
            print(f"\n  ⚠️  Error copying {src.name}: {e}")
            errors += 1

    print(f"  🖼️  {img_folder}/  →  Copied: {copied:,}  |  Skipped: {skipped:,}  |  Errors: {errors:,}")
    grand_total["copied"]  += copied
    grand_total["skipped"] += skipped
    grand_total["errors"]  += errors

# ── Grand summary ─────────────────────────────────────────────
print(f"\n{'='*60}")
print("  ✅  ALL DONE")
print(f"{'='*60}")
print(f"  JSON files processed : {len(json_files)}")
print(f"  Total records        : {grand_total['records']:,}")
print(f"  Total image refs     : {grand_total['img_refs']:,}")
print(f"  Matched on disk      : {grand_total['matched']:,}")
print(f"  Not found on disk    : {grand_total['missing']:,}")
print(f"  Images copied        : {grand_total['copied']:,}")
print(f"  Images skipped       : {grand_total['skipped']:,}  (already existed)")
print(f"  Errors               : {grand_total['errors']:,}")
print(f"\n  Output folder        : {OUTPUT_BASE}")
print(f"{'='*60}")

  MDSM DATA PREP — LOCAL
  Source : C:\Users\User\Downloads\MDSM-public\MDSM-public
  Output : C:\Users\User\Downloads\FYP\mdsm

🔍 Found 9 JSON file(s) in meta/:
   • bbc.json
   • guard_test.json
   • guard_train.json
   • guard_val.json
   • test_fromNYT.json
   • train_fromNYT.json
   • usa_today.json
   • val_fromNYT.json
   • washington_post.json

────────────────────────────────────────────────────────────
  📰 bbc.json
────────────────────────────────────────────────────────────
  Records parsed : 39,109
  💾 Saved → bbc.csv  (6.45 MB)
  Image refs     : 39,109
  Matched on disk: 39,109


  Copying bbc: 100%|██████████| 39109/39109 [00:56<00:00, 695.86img/s]


  🖼️  bbc-image/  →  Copied: 39,109  |  Skipped: 0  |  Errors: 0

────────────────────────────────────────────────────────────
  📰 guard_test.json
────────────────────────────────────────────────────────────
  Records parsed : 15,083
  💾 Saved → guard_test.csv  (2.43 MB)
  Image refs     : 15,083
  Matched on disk: 15,083


  Copying guard_test: 100%|██████████| 15083/15083 [00:21<00:00, 716.96img/s]


  🖼️  guard_test-image/  →  Copied: 15,083  |  Skipped: 0  |  Errors: 0

────────────────────────────────────────────────────────────
  📰 guard_train.json
────────────────────────────────────────────────────────────
  Records parsed : 120,653
  💾 Saved → guard_train.csv  (18.91 MB)
  Image refs     : 120,653
  Matched on disk: 120,653


...

  🖼️  usa_today-image/  →  Copied: 42,885  |  Skipped: 0  |  Errors: 0

────────────────────────────────────────────────────────────
  📰 val_fromNYT.json
────────────────────────────────────────────────────────────
  Records parsed : 17,432
  💾 Saved → val_fromNYT.csv  (2.77 MB)
  Image refs     : 17,432
  Matched on disk: 17,432


  Copying val_fromNYT: 100%|██████████| 17432/17432 [00:32<00:00, 531.75img/s]


  🖼️  val_fromNYT-image/  →  Copied: 17,432  |  Skipped: 0  |  Errors: 0

────────────────────────────────────────────────────────────
  📰 washington_post.json
────────────────────────────────────────────────────────────
  Records parsed : 34,283
  💾 Saved → washington_post.csv  (6.09 MB)
  Image refs     : 34,283
  Matched on disk: 34,283


  Copying washington_post: 100%|██████████| 34283/34283 [01:16<00:00, 448.32img/s]

  🖼️  washington_post-image/  →  Copied: 34,283  |  Skipped: 0  |  Errors: 0

  ✅  ALL DONE
  JSON files processed : 9
  Total records        : 441,423
  Total image refs     : 441,423
  Matched on disk      : 441,423
  Not found on disk    : 0
  Images copied        : 441,423
  Images skipped       : 0  (already existed)
  Errors               : 0

  Output folder        : C:\Users\User\Downloads\FYP\mdsm


In [ ]:
# ============================================================
# Update CSVs only — no image copying
#
# Reads all JSON files in MDSM-public/meta/ and rewrites
# the CSVs in OUTPUT_BASE with the correct format:
#   text | label | local_image_path
#
# Requirements:
#   pip install pandas tqdm
# ============================================================

import os, json, glob
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  CONFIGURE THESE BEFORE RUNNING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Full path to your MDSM-public folder (source JSONs)
DATASET_ROOT = r"C:\Users\User\Downloads\MDSM-public\MDSM-public"   # ← CHANGE THIS

# Folder where your CSVs already live (same place as before)
OUTPUT_BASE  = r"C:\Users\User\Downloads\FYP\mdsm"      # ← CHANGE THIS

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

META_DIR = os.path.join(DATASET_ROOT, "meta")

print("=" * 60)
print("  UPDATE CSVs — text | label | local_image_path")
print("=" * 60)
print(f"  JSON source : {META_DIR}")
print(f"  CSV output  : {OUTPUT_BASE}")

# ── Discover JSON files ───────────────────────────────────────
json_files = sorted(glob.glob(os.path.join(META_DIR, "*.json")))
print(f"\n🔍 Found {len(json_files)} JSON file(s):")
for jf in json_files:
    print(f"   • {os.path.basename(jf)}")

if not json_files:
    raise FileNotFoundError(f"No JSON files found in {META_DIR}.")

# ── Process each JSON ─────────────────────────────────────────
for json_path in json_files:
    source_stem = Path(json_path).stem           # e.g. "washington-post"
    img_folder  = f"{source_stem}-image"         # e.g. "washington-post-image"
    out_csv     = os.path.join(OUTPUT_BASE, f"{source_stem}.csv")

    print(f"\n{'─'*60}")
    print(f"  📰 {os.path.basename(json_path)}")
    print(f"{'─'*60}")

    # Load JSON
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    except Exception as e:
        print(f"  ⚠️  Failed to read: {e} — skipping.")
        continue

    items = data if isinstance(data, list) else data.get("articles", data.get("data", [data]))
    print(f"  Records found  : {len(items):,}")

    # Build rows
    records = []
    for item in tqdm(items, desc="  Building CSV", unit="row", leave=False):
        if not isinstance(item, dict):
            continue
        fake_cls       = item.get("fake_cls", "orig")
        label          = 0 if fake_cls == "orig" else 1
        img_filename   = Path(item.get("image", "")).name
        local_img_path = f"{img_folder}/{img_filename}" if img_filename else ""
        records.append({
            "text":             item.get("text", ""),
            "label":            label,
            "local_image_path": local_img_path,
        })

    df = pd.DataFrame(records)
    df.to_csv(out_csv, index=False, encoding="utf-8-sig")
    csv_mb = os.path.getsize(out_csv) / 1_048_576
    print(f"  ✅ Saved → {os.path.basename(out_csv)}  ({len(df):,} rows, {csv_mb:.2f} MB)")
    print(f"     Label breakdown: {df['label'].value_counts().to_dict()}")

print(f"\n{'='*60}")
print("  ✅  ALL CSVs UPDATED")
print(f"{'='*60}")

  UPDATE CSVs — text | label | local_image_path
  JSON source : C:\Users\User\Downloads\MDSM-public\MDSM-public\meta
  CSV output  : C:\Users\User\Downloads\FYP\mdsm

🔍 Found 9 JSON file(s):
   • bbc.json
   • guard_test.json
   • guard_train.json
   • guard_val.json
   • test_fromNYT.json
   • train_fromNYT.json
   • usa_today.json
   • val_fromNYT.json
   • washington_post.json

────────────────────────────────────────────────────────────
  📰 bbc.json
────────────────────────────────────────────────────────────
  Records found  : 39,109


  ✅ Saved → bbc.csv  (39,109 rows, 5.53 MB)
     Label breakdown: {1: 26312, 0: 12797}

────────────────────────────────────────────────────────────
  📰 guard_test.json
────────────────────────────────────────────────────────────
  Records found  : 15,083


  ✅ Saved → guard_test.csv  (15,083 rows, 2.15 MB)
     Label breakdown: {1: 9282, 0: 5801}

────────────────────────────────────────────────────────────
  📰 guard_train.json
────────────────────────────────────────────────────────────
  Records found  : 120,653


  ✅ Saved → guard_train.csv  (120,653 rows, 16.97 MB)
     Label breakdown: {1: 60412, 0: 60241}

────────────────────────────────────────────────────────────
  📰 guard_val.json
────────────────────────────────────────────────────────────
  Records found  : 15,081


  ✅ Saved → guard_val.csv  (15,081 rows, 2.18 MB)
     Label breakdown: {1: 9301, 0: 5780}

────────────────────────────────────────────────────────────
  📰 test_fromNYT.json
────────────────────────────────────────────────────────────
  Records found  : 17,434


...

  ✅ Saved → train_fromNYT.csv  (139,463 rows, 20.79 MB)
     Label breakdown: {0: 70500, 1: 68963}

────────────────────────────────────────────────────────────
  📰 usa_today.json
────────────────────────────────────────────────────────────
  Records found  : 42,885


  ✅ Saved → usa_today.csv  (42,885 rows, 6.76 MB)
     Label breakdown: {1: 34999, 0: 7886}

────────────────────────────────────────────────────────────
  📰 val_fromNYT.json
────────────────────────────────────────────────────────────
  Records found  : 17,432


  ✅ Saved → val_fromNYT.csv  (17,432 rows, 2.50 MB)
     Label breakdown: {0: 8871, 1: 8561}

────────────────────────────────────────────────────────────
  📰 washington_post.json
────────────────────────────────────────────────────────────
  Records found  : 34,283


  ✅ Saved → washington_post.csv  (34,283 rows, 5.51 MB)
     Label breakdown: {1: 27053, 0: 7230}

  ✅  ALL CSVs UPDATED


: 